# Hidden Commercial Activity Inference

This notebook is the end-to-end version of the hackathon solution. It trains a model that identifies consumer cards whose transaction behaviour looks commercially active, while respecting the key modelling issue: consumer cards are not true negatives. They are an unlabeled pool that may contain hidden entrepreneurs.

The primary jury-facing ranking metric is ROC-AUC, so the final ranking score is `score_auc_optimized`. The calibrated score is used for business segmentation and operational interpretation.

## Problem Framing

Banks often have retail cardholders who behave like small businesses, freelancers, resellers, or self-employed entrepreneurs, but still use consumer products. The task is to infer that hidden commercial activity from transaction behaviour.

The difficult part is label quality. Known business cards are reliable positives. Consumer cards are not reliable negatives because some of them are exactly the hidden commercial users we want to find. This solution therefore uses positive-unlabeled learning to discover reliable negatives, then trains a supervised ensemble for high-quality ranking.

## Pipeline Design

1. Load business, consumer, and merchant-reference transaction data.
2. Add MCC, temporal, channel, geography, and merchant semantics.
3. Aggregate transactions to card-level features.
4. Run PU bagging to avoid treating all consumers as true negatives.
5. Extract reliable negatives from low PU-score consumers.
6. Train LightGBM, CatBoost, and positive-only Isolation Forest.
7. Tune ensemble weights for ROC-AUC on held-out calibration data.
8. Validate on held-out positives and held-out reliable negatives.
9. Score all consumer cards, assign segments, export reason codes and artifacts.

In [ ]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.model_selection import train_test_split

import config
from pipeline.ingest import load_data
from pipeline.mcc_tags import apply_mcc_tags
from pipeline.features import build_features
from pipeline.quality import validate_transactions, validate_feature_frame
from pipeline.pu_learning import run_pu_bagging
from pipeline.reliable_negatives import extract_reliable_negatives
from pipeline.tuning import tune_and_train
from pipeline.validation import validate
from pipeline.scoring import score_consumers
from pipeline.segmentation import assign_segments, print_segment_summary
from pipeline.explainability import run_shap, plot_feature_importance, add_reason_codes
from pipeline.export import export_results
from pipeline.save_models import save_models
from pipeline.reporting import save_run_metadata

warnings.filterwarnings("ignore")

# Keep these values aligned with config.py for the final run.
# For a quick smoke test, temporarily set N_OPTUNA_TRIALS to a small value such as 5.
DATA_DIR = Path(".")
OUT_DIR = Path("./output_notebook")
N_OPTUNA_TRIALS = config.N_OPTUNA_TRIALS
SEED = config.SEED

OUT_DIR.mkdir(parents=True, exist_ok=True)
config.DATA_DIR = DATA_DIR
config.OUT_DIR = OUT_DIR
config.N_OPTUNA_TRIALS = N_OPTUNA_TRIALS
config.SEED = SEED
config.CATBOOST_PARAMS["random_seed"] = SEED
config.ISO_FOREST_PARAMS["random_state"] = SEED

np.random.seed(SEED)
random.seed(SEED)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUT_DIR.resolve()}")
print(f"Optuna trials: {N_OPTUNA_TRIALS}")

## Load Data And Build Features

The feature matrix is built at card level. Metadata fields such as card tier and bank name are kept for reference but intentionally excluded from model features because synthetic data can make them dangerously label-correlated.

In [ ]:
all_txns = load_data(DATA_DIR)
validate_transactions(all_txns, OUT_DIR)

all_txns = apply_mcc_tags(all_txns, config.B2B_MCCS, config.MIXED_MCCS)
card_agg = build_features(all_txns, config.FEATURES)
validate_feature_frame(card_agg, config.FEATURES, OUT_DIR)

biz_df = card_agg[card_agg["label"] == 1].copy().reset_index(drop=True)
cons_df = card_agg[card_agg["label"] == 0].copy().reset_index(drop=True)

print(f"Business cards: {len(biz_df):,}")
print(f"Consumer cards: {len(cons_df):,}")
print(f"Feature count: {len(config.FEATURES):,}")
display(card_agg.head())

## Quick Behavioural Sanity Check

These summaries are not used as final evidence by themselves, but they help verify that the engineered behavioural features separate known business cards from consumers in plausible ways.

In [ ]:
overview_cols = [
    "tx_count", "total_spend", "b2b_ratio", "unique_merchants",
    "mcc_entropy", "merchant_concentration", "business_hours_ratio",
    "offline_ratio", "active_days", "round_large_ratio",
]

label_summary = (
    card_agg.groupby("label")[overview_cols]
    .agg(["mean", "median"])
    .rename(index={0: "consumer_or_unlabeled", 1: "known_business"})
)

display(label_summary)
display(pd.read_csv(OUT_DIR / "data_quality_report.csv"))
display(pd.read_csv(OUT_DIR / "feature_quality_report.csv"))

## Hold Out True Business Positives

Known business cards are split before PU learning and model training. The held-out business cards are used only in validation, which keeps the validation signal cleaner.

In [ ]:
biz_train_idx, biz_test_idx = train_test_split(
    np.arange(len(biz_df)), test_size=0.2, random_state=SEED
)

biz_train_df = biz_df.iloc[biz_train_idx].reset_index(drop=True)
biz_test_df = biz_df.iloc[biz_test_idx].reset_index(drop=True)
X_pos = biz_train_df[config.FEATURES].values

print(f"Business train cards: {len(biz_train_df):,}")
print(f"Business held-out cards: {len(biz_test_df):,}")

## Positive-Unlabeled Learning

Instead of forcing all consumers to be class 0, PU bagging estimates how business-like each consumer card is. The lowest-scoring consumer cards become reliable negatives for supervised training.

In [ ]:
pu_scores = run_pu_bagging(
    biz_train_df,
    cons_df,
    config.FEATURES,
    config.N_BAGS,
    config.BAG_RATIO,
    SEED,
)

(
    reliable_negs_train,
    reliable_negs_test,
    train_df,
    X_train,
    y_train,
    pos_weight,
) = extract_reliable_negatives(
    biz_train_df,
    cons_df,
    pu_scores,
    config.FEATURES,
    config.RELIABLE_NEG_QUANTILE,
    SEED,
)

display(pd.DataFrame({"pu_score": pu_scores}).describe(percentiles=[0.05, 0.2, 0.5, 0.8, 0.95]))

## Train Models

LightGBM is tuned with Optuna for ROC-AUC. CatBoost adds a second tree-boosting view of the data. Isolation Forest is trained on known business cards only, so it contributes a positive-profile anomaly signal.

In [ ]:
lgb_model, catboost_model, iso_model, best_params = tune_and_train(
    X_train=X_train,
    y_train=y_train,
    X_pos=X_pos,
    pos_weight=pos_weight,
    seed=SEED,
    n_trials=N_OPTUNA_TRIALS,
    n_cv_folds=config.N_CV_FOLDS,
    catboost_params=config.CATBOOST_PARAMS,
    iso_params=config.ISO_FOREST_PARAMS,
)

best_params

## Validation And Calibration

Validation uses held-out business positives and held-out reliable negatives. Ensemble weights are tuned for ROC-AUC on a calibration split, then evaluated on a separate split. The tuned raw ensemble is the main ranking score.

In [ ]:
val_results = validate(
    biz_df=biz_test_df,
    reliable_negs_test=reliable_negs_test,
    cons_df=cons_df,
    pu_scores=pu_scores,
    lgb_model=lgb_model,
    catboost_model=catboost_model,
    iso_model=iso_model,
    X_pos=X_pos,
    features=config.FEATURES,
    out_dir=OUT_DIR,
    seed=SEED,
    ensemble_weights=config.ENSEMBLE_WEIGHTS,
)

metrics = pd.read_csv(OUT_DIR / "validation_metrics.csv")
display(metrics.sort_values("roc_auc", ascending=False))
print("Tuned ensemble weights:", val_results["ensemble_weights"])

## Score Consumers, Segment, Explain, Export

`score_auc_optimized` is the ranking score to submit or prioritize for ROC-AUC. `score_calibrated` is used for segment labels because it is easier to interpret as an operational risk/opportunity score.

In [ ]:
cons_scored = score_consumers(
    cons_df=cons_df,
    lgb_model=lgb_model,
    catboost_model=catboost_model,
    iso_model=iso_model,
    iso_ref=val_results["iso_ref"],
    score_calibrator=val_results["score_calibrator"],
    features=config.FEATURES,
    ensemble_weights=val_results["ensemble_weights"],
)

save_models(
    lgb_model,
    catboost_model,
    iso_model,
    iso_ref=val_results["iso_ref"],
    score_calibrator=val_results["score_calibrator"],
    ensemble_weights=val_results["ensemble_weights"],
    out_dir=OUT_DIR,
)

cons_scored = assign_segments(cons_scored, config.SEGMENT_THRESHOLDS)
print_segment_summary(cons_scored, config.SEGMENT_ACTIONS)

run_shap(lgb_model, cons_scored, config.FEATURES, OUT_DIR)
plot_feature_importance(lgb_model, config.FEATURES, OUT_DIR)
cons_scored = add_reason_codes(
    lgb_model, cons_scored, config.FEATURES, top_k=config.N_REASON_CODES
)

export_results(cons_scored, OUT_DIR, config.OUTPUT_COLUMNS)
save_run_metadata(
    OUT_DIR,
    config,
    best_params,
    val_results,
    card_count=len(card_agg),
    consumer_count=len(cons_scored),
    business_count=len(biz_df),
)

scores = pd.read_csv(OUT_DIR / "hidden_entrepreneur_scores.csv")
display(scores.head(20))

## Review Output Artifacts

The exported files are the strongest evidence package for the presentation: validation metrics, ROC curve, PR curve, feature importance, SHAP summary, scored consumer cards, model files, and reproducibility metadata.

In [ ]:
artifact_paths = [
    OUT_DIR / "hidden_entrepreneur_scores.csv",
    OUT_DIR / "validation_metrics.csv",
    OUT_DIR / "feature_separation_report.csv",
    OUT_DIR / "score_distribution_report.csv",
    OUT_DIR / "hard_consumer_stress_report.csv",
    OUT_DIR / "data_quality_report.csv",
    OUT_DIR / "feature_quality_report.csv",
    OUT_DIR / "run_config.json",
    OUT_DIR / "roc_curve.png",
    OUT_DIR / "pr_curve.png",
    OUT_DIR / "feature_importance.png",
    OUT_DIR / "shap_summary.png",
]

artifact_table = pd.DataFrame(
    [{"artifact": str(path), "exists": path.exists()} for path in artifact_paths]
)
display(artifact_table)

for image_name in ["roc_curve.png", "pr_curve.png", "feature_importance.png", "shap_summary.png"]:
    path = OUT_DIR / image_name
    if path.exists():
        print(image_name)
        display(Image(filename=str(path)))

## Real-Data Accuracy Notes

The solution is designed to rank real consumer cards by commercial-likelihood, not merely to maximize synthetic-data accuracy. The most important safeguards are PU learning, held-out validation, metadata leakage control, ROC-AUC optimized ensemble weights, data quality gates, calibrated business segmentation, and explainable reason codes.

Near-perfect validation should be treated skeptically: it usually means the synthetic data or selected reliable negatives are very separable, not that real-world performance will be 100%. The biggest remaining real-world risks are label shift, merchant taxonomy changes, seasonality, and synthetic-data over-separation. In production, the next step would be time-based backtesting, post-campaign feedback labels, and monitoring score drift by month, bank, tier, and acquisition channel.